# IntelliDoc AI — Notebook 4: CORD Receipt Understanding & Verification

**Objective:**
1. Validate and profile the **CORD v2 (Consolidated Receipt Dataset)**.
2. Inspect fine-grained line-item categories: `menu.nm`, `menu.cnt`, `menu.price`, `sub_total.tax_price`, `total.total_price`.
3. Apply IntelliDoc's deterministic **arithmetic verification invariant** on retail receipts (`quantity × unit_price = line_total` and `subtotal + tax = total`).

---
### Dataset Overview:
- **Source:** Clova AI (Naver CORD v2)
- **Format:** Scanned retail receipts (PNG/JPG) + Quadrilateral bounding box annotations (JSON)
- **Primary Use in IntelliDoc:** Line-item arithmetic cross-auditing and invoice/receipt multimodal understanding.

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import pandas as pd

# Check if running on Google Colab or local environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    is_colab = True
except ImportError:
    is_colab = False

# Set CORD root path (works on both Google Colab and local PC)
if is_colab:
    cord_dir = Path("/content/drive/MyDrive/Intellidoc_AI/datasets_public/cord")
else:
    cord_dir = Path("../datasets_public/cord").resolve()

print(f"CORD Dataset Root: {cord_dir}")
print(f"Exists: {cord_dir.exists()}")

In [ ]:
# Load statistical profile generated by validate_cord.py
summary_path = cord_dir / "cord_summary.json"
if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        stats = json.load(f)
    print(f"Total Paired Receipts: {stats['total_receipts']}")
    print(f"Splits: {list(stats['splits'].keys())}")
    df_cats = pd.DataFrame(list(stats['categories_distribution'].items()), columns=['Category', 'Count'])
    display(df_cats.head(15))
else:
    print("Run validate_cord.py to generate summary stats.")

In [ ]:
# Inspect and visualize sample receipt with bounding boxes
test_json_dir = cord_dir / "test" / "json"
test_img_dir = cord_dir / "test" / "image"

sample_files = list(test_json_dir.glob("*.json"))[:3]

for sf in sample_files:
    with open(sf, "r", encoding="utf-8") as f:
        ann = json.load(f)
    
    img_path = test_img_dir / f"{sf.stem}.png"
    if not img_path.exists():
        img_path = test_img_dir / f"{sf.stem}.jpg"
    
    if img_path.exists():
        img = Image.open(img_path).convert("RGB")
        draw = ImageDraw.Draw(img)
        
        print(f"=== Receipt: {sf.name} ({img.width}x{img.height}) ===")
        for line in ann.get("valid_line", []):
            cat = line.get("category", "")
            for word in line.get("words", []):
                quad = word.get("quad", {})
                if quad:
                    pts = [(quad['x1'], quad['y1']), (quad['x2'], quad['y2']),
                           (quad['x3'], quad['y3']), (quad['x4'], quad['y4'])]
                    # Color code: total=red, menu=blue, subtotal=green
                    color = "blue" if "menu" in cat else ("red" if "total" in cat else "green")
                    draw.polygon(pts, outline=color, width=2)
        
        plt.figure(figsize=(8, 10))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"CORD Receipt: {sf.stem}")
        plt.show()
        break

### Step 2: Deterministic Receipt Arithmetic Audit
Next, we extract all line items (`menu.price`) and compare against `total.total_price` to verify zero-hallucination mathematical consistency.

In [ ]:
# Deterministic arithmetic parser on CORD receipts
import re

def clean_num(val_str):
    clean = re.sub(r'[^0-9.]', '', val_str.replace(',', '.'))
    try:
        return float(clean)
    except Exception:
        return None

audit_results = []
for sf in list(test_json_dir.glob("*.json"))[:50]:
    with open(sf, "r", encoding="utf-8") as f:
        ann = json.load(f)
    
    line_prices = []
    stated_total = None
    stated_tax = None
    
    for line in ann.get("valid_line", []):
        cat = line.get("category", "")
        text = " ".join([w["text"] for w in line.get("words", []) if w.get("is_key") == 0])
        
        if cat == "menu.price":
            p = clean_num(text)
            if p is not None:
                line_prices.append(p)
        elif cat == "total.total_price":
            stated_total = clean_num(text)
        elif cat == "sub_total.tax_price":
            stated_tax = clean_num(text)
    
    if line_prices and stated_total:
        calc_subtotal = sum(line_prices)
        diff = abs((calc_subtotal + (stated_tax or 0)) - stated_total)
        is_match = diff <= 1.0
        audit_results.append({
            "file": sf.name,
            "items_count": len(line_prices),
            "calc_sum": calc_subtotal,
            "stated_tax": stated_tax,
            "stated_total": stated_total,
            "diff": diff,
            "status": "MATCH" if is_match else "REVIEW"
        })

df_audit = pd.DataFrame(audit_results)
print(f"Evaluated {len(df_audit)} CORD test receipts with IntelliDoc Arithmetic Engine:")
display(df_audit.head(10))
print("Audit Distribution:")
print(df_audit['status'].value_counts())